# **LAB ASSIGNMENT - 7**

**Tokenization + Dataset Setup**

In [1]:
# Training data
documents = [
    (["taipei", "taiwan"], "c"),
    (["macao", "taiwan", "shanghai"], "c"),
    (["japan", "sapporo"], "j"),
    (["sapporo", "osaka", "taiwan"], "j")
]

# Test document
test_doc = ["taiwan", "taiwan", "sapporo"]

print("Training Data:", documents)
print("Test Document:", test_doc)

Training Data: [(['taipei', 'taiwan'], 'c'), (['macao', 'taiwan', 'shanghai'], 'c'), (['japan', 'sapporo'], 'j'), (['sapporo', 'osaka', 'taiwan'], 'j')]
Test Document: ['taiwan', 'taiwan', 'sapporo']


**Vocabulary Creation**

In [2]:
vocab = set()

for doc, _ in documents:
    vocab.update(doc)

vocab = list(vocab)
print("Vocabulary:", vocab)
print("Vocabulary Size:", len(vocab))

Vocabulary: ['shanghai', 'macao', 'taipei', 'japan', 'sapporo', 'osaka', 'taiwan']
Vocabulary Size: 7


**Compute Prior Probabilities**

In [3]:
from collections import Counter

class_counts = Counter([cls for _, cls in documents])
total_docs = len(documents)

priors = {cls: count/total_docs for cls, count in class_counts.items()}

print("Class Counts:", class_counts)
print("Prior Probabilities:", priors)

Class Counts: Counter({'c': 2, 'j': 2})
Prior Probabilities: {'c': 0.5, 'j': 0.5}


**Word Counts per Class**

In [4]:
word_counts = {
    "c": Counter(),
    "j": Counter()
}

for doc, cls in documents:
    word_counts[cls].update(doc)

print("Word Counts per Class:")
print(word_counts)

Word Counts per Class:
{'c': Counter({'taiwan': 2, 'taipei': 1, 'macao': 1, 'shanghai': 1}), 'j': Counter({'sapporo': 2, 'japan': 1, 'osaka': 1, 'taiwan': 1})}


**Total Words per Class**

In [5]:
total_words = {
    cls: sum(word_counts[cls].values())
    for cls in word_counts
}

print("Total Words per Class:", total_words)

Total Words per Class: {'c': 5, 'j': 5}


**Conditional Probabilities (Laplace Smoothing)**

In [6]:
import pandas as pd

cond_prob = {
    "c": {},
    "j": {}
}

V = len(vocab)

for cls in ["c", "j"]:
    for word in vocab:
        cond_prob[cls][word] = (word_counts[cls][word] + 1) / (total_words[cls] + V)

# Display nicely
df = pd.DataFrame(cond_prob)
print(df)

                 c         j
shanghai  0.166667  0.083333
macao     0.166667  0.083333
taipei    0.166667  0.083333
japan     0.083333  0.166667
sapporo   0.083333  0.250000
osaka     0.083333  0.166667
taiwan    0.250000  0.166667


**Compute Probability of Test Document**

In [7]:
import math

def compute_prob(test_doc, cls):
    prob = math.log(priors[cls])  # log for stability

    for word in test_doc:
        if word in vocab:
            prob += math.log(cond_prob[cls][word])
        else:
            # unseen word
            prob += math.log(1 / (total_words[cls] + V))

    return prob

prob_c = compute_prob(test_doc, "c")
prob_j = compute_prob(test_doc, "j")

print("Log Probability for class c:", prob_c)
print("Log Probability for class j:", prob_j)

Log Probability for class c: -5.950642552587727
Log Probability for class j: -5.662960480135946


**Final Prediction**

In [8]:
if prob_c > prob_j:
    print("Test document belongs to class: c")
else:
    print("Test document belongs to class: j")

Test document belongs to class: j
